In [15]:
import hashlib
import json
import os
import pickle
import re
import time
import uuid
from datetime import UTC, datetime
from functools import lru_cache
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import bm25s
import numpy as np
import pandas as pd
import stopwordsiso as stopwords
from dotenv import load_dotenv
from IPython.display import display
from sentence_transformers import SentenceTransformer
from sklearn.metrics import ndcg_score
from sklearn.metrics.pairwise import haversine_distances, linear_kernel
from sklearn.model_selection import ParameterGrid
from sklearn.preprocessing import minmax_scale

DATASET_VERSION = "v1"
MODEL_VERSION = "hybrid_bm25_dense_geo_v1"

ROOT_DIR = Path.cwd().parent
RAW_DIR = ROOT_DIR / "data" / "raw"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
INDEX_DIR = ROOT_DIR / "data" / "indexes"
BENCHMARK_DIR = ROOT_DIR / "data" / "benchmarks"
AUDIT_DIR = ROOT_DIR / "data" / "audit"

# load environment variables from .env file
load_dotenv(ROOT_DIR / ".env")

for directory in [PROCESSED_DIR, INDEX_DIR, BENCHMARK_DIR, AUDIT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

TOP_K = 10
RANDOM_SEED = 31

In [16]:
REQUIRED_COLUMNS = [
    "Operation_Name_English",
    "Operation_Summary_English",
    "Operation_Name_Programme_Language",
    "Operation_Summary_Programme_Language",
    "Country",
    "CountryCode",
    "Location_Indicator_latitude_longitude",
    "NUTS3_Label",
    "LAU_Labels",
    "Operation_Unique_Identifier",
    "Programme_Name",
    "Fund_Name",
    "Category_Label",
    "Specific_Objective_Label",
    "Policy_Objective_Label",
    "Total_Eligible_Expenditure_amount",
    "Project_EU_Budget",
    "InfoRegio_URL",
]

csv_files = sorted(RAW_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {RAW_DIR}")

schema_report = []
for csv_file in csv_files:
    header = pd.read_csv(csv_file, nrows=0).columns.tolist()
    missing = sorted(set(REQUIRED_COLUMNS) - set(header))
    schema_report.append(
        {
            "file": csv_file.name,
            "column_count": len(header),
            "missing_required_columns": missing,
        }
    )

schema_df = pd.DataFrame(schema_report)

# check if any files are missing required columns
files_with_issues = schema_df[schema_df["missing_required_columns"].apply(len) > 0]
if not files_with_issues.empty:
    print("The following files are missing required columns:")
    for _, row in files_with_issues.iterrows():
        print(f"- {row['file']}: Missing columns: {', '.join(row['missing_required_columns'])}")
else:
    print("All files contain the required columns.")

schema_df.head()

All files contain the required columns.


,file,column_count,missing_required_columns
0,latest_AT-pp21-27-latest.csv,42,[]
1,latest_BE-pp21-27-latest.csv,42,[]
2,latest_BG-pp21-27-latest.csv,42,[]
3,latest_CY-pp21-27-latest.csv,42,[]
4,latest_CZ-pp21-27-latest.csv,42,[]


In [17]:
def clean_text(value):
    if pd.isna(value):
        return ""
    text = str(value)
    # replace multiple whitespace characters with a single space
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def clean_label(value):
    text = clean_text(value)
    return text if text else np.nan


def parse_money(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().replace(" ", "").replace(",", "")
    number = pd.to_numeric(text, errors="coerce")
    return float(number) if pd.notna(number) else np.nan


def parse_first_lat_lon(value):
    text = clean_text(value)
    if not text:
        return (np.nan, np.nan)
    first_location = text.split("|")[0]
    parts = [part.strip() for part in first_location.split(",")]
    if len(parts) != 2:
        return (np.nan, np.nan)
    latitude = pd.to_numeric(parts[0], errors="coerce")
    longitude = pd.to_numeric(parts[1], errors="coerce")
    return (
        float(latitude) if pd.notna(latitude) else np.nan,
        float(longitude) if pd.notna(longitude) else np.nan,
    )


def combine_text_fields(row):
    fields = [
        row.get("Operation_Name_English", ""),
        row.get("Operation_Summary_English", ""),
        row.get("Operation_Name_Programme_Language", ""),
        row.get("Operation_Summary_Programme_Language", ""),
    ]
    cleaned_fields = [clean_text(field) for field in fields]
    non_empty_fields = [field for field in cleaned_fields if field]
    return clean_text(". ".join(non_empty_fields))


processed_path = PROCESSED_DIR / f"eu_projects_{DATASET_VERSION}.parquet"

# check if the processed file already exists to avoid reprocessing
if processed_path.exists():
    processed_df = pd.read_parquet(processed_path)
else:
    frames = []
    for csv_file in csv_files:
        frame = pd.read_csv(csv_file, low_memory=False)
        frame["source_file"] = csv_file.name
        frames.append(frame)

    raw_df = pd.concat(frames, ignore_index=True)

    for column in raw_df.select_dtypes(include=["object", "str"]).columns:
        raw_df[column] = raw_df[column].map(clean_text)

    processed_df = raw_df.copy()
    processed_df["record_id"] = processed_df["Operation_Unique_Identifier"].mask(
        processed_df["Operation_Unique_Identifier"].str.len() == 0,
        pd.Series(processed_df.index.astype(str), index=processed_df.index),
    )
    processed_df["CountryCode"] = processed_df["CountryCode"].map(
        lambda value: clean_text(value).upper()
    )
    processed_df["Country"] = processed_df["Country"].map(clean_label)
    processed_df["NUTS3_Label"] = processed_df["NUTS3_Label"].map(clean_label)
    processed_df["LAU_Labels"] = processed_df["LAU_Labels"].map(clean_label)
    processed_df["Total_Eligible_Expenditure_amount"] = processed_df[
        "Total_Eligible_Expenditure_amount"
    ].map(parse_money)
    processed_df["Project_EU_Budget"] = processed_df["Project_EU_Budget"].map(parse_money)
    processed_df[["latitude", "longitude"]] = processed_df[
        "Location_Indicator_latitude_longitude"
    ].apply(lambda value: pd.Series(parse_first_lat_lon(value)))
    processed_df["search_text"] = processed_df.apply(combine_text_fields, axis=1)

    # filter out records with empty search_text
    processed_df = processed_df[processed_df["search_text"].str.len() > 0].reset_index(drop=True)

    processed_df["dataset_version"] = DATASET_VERSION
    processed_df.to_parquet(processed_path, index=False)

print(f"Total records: {len(processed_df)}, Unique countries: {processed_df['Country'].nunique()}")

processed_df[["record_id", "Country", "Programme_Name", "Fund_Name", "search_text"]].head()

Total records: 124988, Unique countries: 26


,record_id,Country,Programme_Name,Fund_Name,search_text
0,https://linkedopendata.eu/entity/Q7500632,Austria,Employment - AT - ESF+/JTF,European Social Fund Plus,Basic education at the Volkshochschule Salzbur...
1,https://linkedopendata.eu/entity/Q7500999,Austria,Investments in employment and growth 2021-27 –...,Just Transition Fund,Spielberg production campus. ecomaster technol...
2,https://linkedopendata.eu/entity/Q7500933,Austria,Investments in employment and growth 2021-27 –...,Just Transition Fund,JTF creates future space Salzkammergut Nord. E...
3,https://linkedopendata.eu/entity/Q7500699,Austria,Employment - AT - ESF+/JTF,European Social Fund Plus,Continue learning with the School Success Asso...
4,https://linkedopendata.eu/entity/Q7500704,Austria,Employment - AT - ESF+/JTF,European Social Fund Plus,Computer courses 2024 in Güssing. The training...


In [18]:
EU_LANGUAGE_CODES = [
    "bg",
    "cs",
    "da",
    "de",
    "el",
    "en",
    "es",
    "et",
    "fi",
    "fr",
    "hr",
    "hu",
    "it",
    "lt",
    "lv",
    "mt",
    "nl",
    "pl",
    "pt",
    "ro",
    "sk",
    "sl",
    "sv",
]

# create a set of stop words for all EU languages
STOP_WORDS = set()
for language_code in EU_LANGUAGE_CODES:
    STOP_WORDS.update(stopwords.stopwords(language_code))


def tokenize(text):
    # keep accented characters and remove punctuation, numbers and single character tokens
    tokens = re.findall(r"[\wÀ-ÿ]+", clean_text(text).lower())
    # filter out stop words and tokens of length 1 or less
    return [token for token in tokens if token not in STOP_WORDS and len(token) > 1]


processed_df["bm25_tokens"] = processed_df["search_text"].map(tokenize)
processed_df["bm25_text"] = processed_df["bm25_tokens"].map(" ".join)

print(
    f"Average search_text length: {processed_df['search_text'].str.len().mean():.2f}, "
    f"Average bm25_text length: {processed_df['bm25_text'].str.len().mean():.2f}"
)

processed_df[["search_text", "bm25_text"]].head(3)

Average search_text length: 2159.61, Average bm25_text length: 1635.17


,search_text,bm25_text
0,Basic education at the Volkshochschule Salzbur...,basic education volkshochschule salzburg basic...
1,Spielberg production campus. ecomaster technol...,spielberg production campus ecomaster technolo...
2,JTF creates future space Salzkammergut Nord. E...,jtf creates future space salzkammergut nord en...


In [19]:
def corpus_texts_hash(texts):
    payload = json.dumps(texts, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def min_max_scale(values):
    # use sklearn to convert scores to a 0-1 range before combining retrieval signals
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return values
    return np.nan_to_num(minmax_scale(values))


BM25_INDEX_NAME = "bm25s"
BM25_K1 = 1.5
BM25_B = 0.75
BM25_METHOD = "atire"
BM25_IDF_METHOD = "lucene"


def build_bm25_index(documents):
    # bm25s precomputes a sparse inverted index, so query-time scoring only touches
    # documents containing query terms instead of scanning every document/token pair
    index = bm25s.BM25(k1=BM25_K1, b=BM25_B, method=BM25_METHOD, idf_method=BM25_IDF_METHOD)
    index.index(documents, show_progress=True)
    return index


def bm25_scores(query):
    query_tokens = tokenize(query)
    if not query_tokens:
        return np.zeros(len(processed_df))
    index = bm25_index
    if index is None:
        raise RuntimeError("BM25 index does not exist")
    return index.get_scores(query_tokens).astype(float, copy=False)


BM25_INDEX_PATH = INDEX_DIR / f"{BM25_INDEX_NAME}_{DATASET_VERSION}.pkl"
bm25_documents = processed_df["bm25_tokens"].tolist()

if BM25_INDEX_PATH.exists():
    with open(BM25_INDEX_PATH, "rb") as file:
        bm25_index = pickle.load(file)

    print(f"BM25 index loaded from {BM25_INDEX_PATH}")
else:
    bm25_index = build_bm25_index(bm25_documents)

    with open(BM25_INDEX_PATH, "wb") as file:
        pickle.dump(bm25_index, file)

    print(f"BM25 index saved to {BM25_INDEX_PATH}")

BM25 index loaded from /Users/riccardo/Projects/luiss-xai/data/indexes/bm25s_v1.pkl


In [20]:
DENSE_CANDIDATES = {
    # https://huggingface.co/sentence-transformers/all-mpnet-base-v2
    "sbert_mpnet": "sentence-transformers/all-mpnet-base-v2",
    # https://huggingface.co/intfloat/multilingual-e5-large
    "multilingual_e5": "intfloat/multilingual-e5-large",
}
# multilingual_e5 is the default dense candidate for demonstration purposes
DEFAULT_DENSE_CANDIDATE = "multilingual_e5"


def dense_texts_for_model(texts, candidate_name):
    # E5 models are trained with explicit input prefixes: `passage:` for indexed documents
    # and `query:` for search queries. Other dense models use the raw text directly
    if candidate_name == "multilingual_e5":
        return [f"passage: {text}" for text in texts]
    return texts


def dense_query_for_model(query, candidate_name):
    if candidate_name == "multilingual_e5":
        return f"query: {query}"
    return query


def dense_cache_path(candidate_name):
    return INDEX_DIR / f"{candidate_name}_{DATASET_VERSION}.pkl"


def load_cached_embeddings(candidate_name, model_name, texts_hash):
    cache_path = dense_cache_path(candidate_name)
    if not cache_path.exists():
        return None

    with open(cache_path, "rb") as file:
        cached_index = pickle.load(file)

    if cached_index.get("model_name") != model_name:
        print(f"Ignoring dense cache for {candidate_name}: model changed.")
        return None
    if cached_index.get("dataset_version") != DATASET_VERSION:
        print(f"Ignoring dense cache for {candidate_name}: dataset version changed.")
        return None
    if cached_index.get("texts_hash") != texts_hash:
        print(f"Ignoring dense cache for {candidate_name}: corpus text changed.")
        return None

    print(f"Loaded cached embeddings for {candidate_name} from {cache_path}.")
    return cached_index["embeddings"]


def save_cached_embeddings(candidate_name, model_name, texts_hash, embeddings):
    cache_path = dense_cache_path(candidate_name)
    with open(cache_path, "wb") as file:
        pickle.dump(
            {
                "name": candidate_name,
                "model_name": model_name,
                "dataset_version": DATASET_VERSION,
                "texts_hash": texts_hash,
                "embeddings": embeddings,
            },
            file,
        )
    print(f"Saved embeddings for {candidate_name} to {cache_path}.")


def build_dense_index(candidate_name, model_name, texts):
    texts_hash = corpus_texts_hash(texts)
    embeddings = load_cached_embeddings(candidate_name, model_name, texts_hash)
    model = SentenceTransformer(model_name)

    # if embeddings are not cached, compute and cache them
    if embeddings is None:
        embeddings = model.encode(
            dense_texts_for_model(texts, candidate_name),
            batch_size=32,
            normalize_embeddings=True,
            show_progress_bar=True,
        )
        save_cached_embeddings(candidate_name, model_name, texts_hash, embeddings)

    return {
        "name": candidate_name,
        "model_name": model_name,
        "model": model,
        "embeddings": embeddings,
    }


corpus_texts = processed_df["search_text"].tolist()
dense_indexes = {}

for candidate_name, model_name in DENSE_CANDIDATES.items():
    dense_indexes[candidate_name] = build_dense_index(candidate_name, model_name, corpus_texts)

Loaded cached embeddings for sbert_mpnet from /Users/riccardo/Projects/luiss-xai/data/indexes/sbert_mpnet_v1.pkl.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded cached embeddings for multilingual_e5 from /Users/riccardo/Projects/luiss-xai/data/indexes/multilingual_e5_v1.pkl.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [21]:
NOMINATIM_SEARCH_URL = "https://nominatim.openstreetmap.org/search"
NOMINATIM_USER_AGENT = "luiss-xai-geocoder/1.0"
NOMINATIM_EMAIL = str(os.getenv("NOMINATIM_EMAIL"))
_last_nominatim_request_at = 0.0


# Nominatim has usage policies that require at most 1 request per second
# and a valid User-Agent header
@lru_cache(maxsize=256)
def search_nominatim(query):
    global _last_nominatim_request_at

    params = {
        "q": query,
        "format": "jsonv2",
        "addressdetails": 1,
        "limit": 1,
        "accept-language": "en",
    }
    if NOMINATIM_EMAIL:
        params["email"] = NOMINATIM_EMAIL

    elapsed = time.monotonic() - _last_nominatim_request_at
    if elapsed < 1:
        time.sleep(1 - elapsed)

    url = f"{NOMINATIM_SEARCH_URL}?{urlencode(params)}"
    request = Request(url, headers={"User-Agent": NOMINATIM_USER_AGENT})
    _last_nominatim_request_at = time.monotonic()

    try:
        with urlopen(request, timeout=10) as response:
            data = json.load(response)
            return data if isinstance(data, list) else []
    except (HTTPError, URLError, TimeoutError, json.JSONDecodeError):
        return []


# this function takes a location text, attempts to geocode it using Nominatim
# and returns a dictionary with the geocoding results
def geocode_location(location_text):
    query = clean_text(location_text)
    if not query:
        return {}

    metadata = {"raw_location": location_text}
    if re.fullmatch(r"[A-Za-z]{2}", query):
        metadata["country_code"] = query.upper()
        # this covers 100% of the corpus
        return metadata

    results = search_nominatim(query)
    if not results:
        return metadata

    result = results[0]
    try:
        metadata["latitude"] = float(result["lat"])
        metadata["longitude"] = float(result["lon"])
    except (KeyError, TypeError, ValueError):
        return metadata

    address = result.get("address", {})
    country_code = address.get("country_code")
    if country_code:
        metadata["country_code"] = country_code.upper()
    if result.get("display_name"):
        metadata["display_name"] = result["display_name"]
    return metadata


EARTH_RADIUS_KM = 6371.0
GEOGRAPHY_COUNTRY_MATCH_WEIGHT = 0.7
GEOGRAPHY_DISTANCE_WEIGHT = 0.3
GEOGRAPHY_DISTANCE_DECAY_KM = 500
GEOGRAPHY_COORDINATE_COLUMNS = ["latitude", "longitude"]


# scores how geographically relevant each project is to the query location.
# country matches contribute most of the score, while nearby coordinates add a smaller
# distance boost that fades out over the configured decay radius
def geography_scores(location_metadata, records):
    if not location_metadata:
        return np.zeros(len(records))
    scores = np.zeros(len(records))
    query_country = location_metadata.get("country_code")
    if query_country:
        scores += (records["CountryCode"].fillna("").str.upper() == query_country).astype(
            float
        ) * GEOGRAPHY_COUNTRY_MATCH_WEIGHT
    coordinate_columns = set(GEOGRAPHY_COORDINATE_COLUMNS)
    has_query_coordinates = coordinate_columns.issubset(location_metadata)
    has_record_coordinates = coordinate_columns.issubset(records.columns)
    if has_query_coordinates and has_record_coordinates:
        record_coordinates = records[GEOGRAPHY_COORDINATE_COLUMNS].apply(
            pd.to_numeric, errors="coerce"
        )
        valid_coordinates = record_coordinates.notna().all(axis=1).to_numpy()
        distance_scores = np.zeros(len(records))

        if valid_coordinates.any():
            query_coordinates = np.radians(
                [[location_metadata["latitude"], location_metadata["longitude"]]]
            )
            candidate_coordinates = np.radians(
                record_coordinates.loc[valid_coordinates].to_numpy(dtype=float)
            )
            distances = (
                haversine_distances(query_coordinates, candidate_coordinates)[0] * EARTH_RADIUS_KM
            )
            distance_scores[valid_coordinates] = np.clip(
                1 - distances / GEOGRAPHY_DISTANCE_DECAY_KM, 0, 1
            )
        scores += distance_scores * GEOGRAPHY_DISTANCE_WEIGHT
    return np.clip(scores, 0, 1)


# sample usage of the geocoding function
print(geocode_location("LUISS Viale Romania, Rome, Italy"))

{'raw_location': 'LUISS Viale Romania, Rome, Italy', 'latitude': 41.9244494, 'longitude': 12.4941045, 'country_code': 'IT', 'display_name': 'Libera Università Internazionale degli Studi Sociali (LUISS) Guido Carli, Viale Romania, Parioli, Municipio Roma II, Rome, Roma Capitale, Lazio, 00197, Italy'}


In [22]:
CONFIDENCE_WEIGHT_GRID = [
    {"semantic": [0.70], "keyword": [0.20], "geographic": [0.10]},
    {"semantic": [0.60], "keyword": [0.30], "geographic": [0.10]},
    {"semantic": [0.60], "keyword": [0.25], "geographic": [0.15]},
    {"semantic": [0.50], "keyword": [0.35], "geographic": [0.15]},
    {"semantic": [0.50], "keyword": [0.30], "geographic": [0.20]},
]
CONFIDENCE_WEIGHT_CANDIDATES = list(ParameterGrid(CONFIDENCE_WEIGHT_GRID))
DEFAULT_CONFIDENCE_WEIGHTS = CONFIDENCE_WEIGHT_CANDIDATES[0]


# encodes the user query with the selected dense model and scores it against all
# project embeddings
def dense_scores(query, dense_index):
    candidate_name = dense_index["name"]
    query_embedding = dense_index["model"].encode(
        [dense_query_for_model(query, candidate_name)],
        normalize_embeddings=True,
    )
    return linear_kernel(dense_index["embeddings"], query_embedding).ravel()


# combines BM25, semantic and geographic scores using the selected confidence weights.
# if geography is unavailable, the active weights are rescaled
def confidence_scores(semantic_score, keyword_score, geographic_score=None, weights=None):
    weights = DEFAULT_CONFIDENCE_WEIGHTS if weights is None else weights
    score_components = {"semantic": semantic_score, "keyword": keyword_score}
    active_weights = {"semantic": weights["semantic"], "keyword": weights["keyword"]}
    if geographic_score is None:
        total_weight = sum(active_weights.values())
    else:
        score_components["geographic"] = geographic_score
        active_weights["geographic"] = weights["geographic"]
        total_weight = sum(active_weights.values())

    weighted_score = np.zeros_like(semantic_score, dtype=float)
    for score_name, weight in active_weights.items():
        weighted_score += score_components[score_name] * weight
    return weighted_score / total_weight


def retrieve_similar_projects(
    # project idea or project description to search with
    query,
    location_text="",
    top_k=TOP_K,
    dense_candidate=DEFAULT_DENSE_CANDIDATE,
    # dictionary controlling how semantic, keyword and geographic scores are combined
    confidence_weights=None,
    # boolean mask used to exclude rows from ranking
    exclusion_mask=None,
):
    location_metadata = geocode_location(location_text)

    # score lexical overlap with BM25 then scale to a comparable 0-1 range
    raw_keyword_scores = bm25_scores(query)
    keyword_score = min_max_scale(raw_keyword_scores)

    # score semantic similarity with the selected dense embedding model
    selected_dense_index = dense_indexes[dense_candidate]
    raw_semantic_scores = dense_scores(query, selected_dense_index)
    semantic_score = min_max_scale(raw_semantic_scores)

    geographic_score = (
        geography_scores(location_metadata, processed_df) if location_metadata else None
    )
    confidence = confidence_scores(
        semantic_score, keyword_score, geographic_score, weights=confidence_weights
    )
    if exclusion_mask is not None:
        confidence = confidence.copy()
        confidence[np.asarray(exclusion_mask, dtype=bool)] = -np.inf

    # select the top-k projects
    ranked_indices = np.argsort(confidence)[::-1][:top_k]
    result_columns = [
        "Operation_Unique_Identifier",
        "Programme_Name",
        "Fund_Name",
        "Category_Label",
        "Specific_Objective_Label",
        "Policy_Objective_Label",
        "Total_Eligible_Expenditure_amount",
        "Project_EU_Budget",
        "Country",
        "NUTS3_Label",
        "LAU_Labels",
        "InfoRegio_URL",
    ]
    results = processed_df.iloc[ranked_indices][result_columns + ["search_text"]].copy()

    results["semantic_score"] = semantic_score[ranked_indices]
    results["keyword_score"] = keyword_score[ranked_indices]
    results["geographic_score"] = (
        geographic_score[ranked_indices] if geographic_score is not None else np.nan
    )
    results["confidence"] = confidence[ranked_indices]
    results["rank"] = range(1, len(results) + 1)

    return results[
        ["rank", "confidence", "semantic_score", "keyword_score", "geographic_score"]
        + result_columns
        + ["search_text"]
    ]


example_query = (
    "Transform an empty train station into a public civic hub for climate change "
    "adaptation, biodiversity protection and cultural heritage regeneration."
)
example_results = retrieve_similar_projects(example_query, location_text="Rovereto, Italy", top_k=5)
example_results

,rank,confidence,semantic_score,keyword_score,geographic_score,Operation_Unique_Identifier,Programme_Name,Fund_Name,Category_Label,Specific_Objective_Label,Policy_Objective_Label,Total_Eligible_Expenditure_amount,Project_EU_Budget,Country,NUTS3_Label,LAU_Labels,InfoRegio_URL,search_text
83126,1,0.999858,1.000000,1.000000,0.998578,https://linkedopendata.eu/entity/Q7420768,European Urban Initiative 2021-2027,European Regional Development Fund,,,Europe closer to citizens,6247121.88,4997697.50,Italy,Trentino,Rovereto,NaN,S4T. The alpine town of Rovereto and its funct...
28040,2,0.645103,0.879349,0.147794,0.000000,https://linkedopendata.eu/entity/Q7414233,Integrated Regional Programme - CZ - ERDF,European Regional Development Fund,"Nature and biodiversity protection, natural he...",RSO5.2,Europe closer to citizens,121650.00,115567.50,Czechia,Central Bohemian Region,Bludov,NaN,Public space at the fire station. Improving pu...
27324,3,0.643775,0.907129,0.043924,0.000000,https://linkedopendata.eu/entity/Q7411440,Integrated Regional Programme - CZ - ERDF,European Regional Development Fund,Water management and water resource conservati...,RSO2.7,"Greener, carbon-free Europe",296772.96,207741.07,Czechia,South Moravian Region,Kuřim,NaN,Revitalization of public space in the Nádražní...
21365,4,0.636625,0.898525,0.036600,0.003380,https://linkedopendata.eu/entity/Q7411372,Integrated Regional Programme - CZ - ERDF,European Regional Development Fund,Renewable energy: solar|Water management and w...,RSO2.7,"Greener, carbon-free Europe",2465693.25,1725985.28,Czechia,Vysočina Region,Telč,NaN,Revitalization of the area of the former winte...
22508,5,0.635648,0.858689,0.172830,0.000000,https://linkedopendata.eu/entity/Q7411443,Integrated Regional Programme - CZ - ERDF,European Regional Development Fund,Water management and water resource conservati...,RSO2.7,"Greener, carbon-free Europe",1460220.63,1241187.54,Czechia,Moravian-Silesian Region,Třinec,NaN,Regeneration of public space above Hradčany. I...


In [23]:
BENCHMARK_SIZE = min(200, len(processed_df))
# for the benchmark, we select a random sample of projects with longer search texts
benchmark_candidates = processed_df[processed_df["search_text"].str.len().gt(80)].sample(
    n=BENCHMARK_SIZE, random_state=RANDOM_SEED
)


# remove any label clues from the search text for more realistic queries
def strip_label_clues(text, row):
    stripped_text = clean_text(text)
    label_fields = [
        "Programme_Name",
        "Fund_Name",
        "Category_Label",
        "Specific_Objective_Label",
        "Policy_Objective_Label",
        "Country",
        "NUTS3_Label",
        "LAU_Labels",
    ]

    for field in label_fields:
        value = clean_text(row.get(field))
        if value:
            stripped_text = re.sub(re.escape(value), " ", stripped_text, flags=re.IGNORECASE)

    return clean_text(stripped_text)


benchmark_df = benchmark_candidates[
    [
        "record_id",
        "CountryCode",
        "Programme_Name",
        "Fund_Name",
        "Category_Label",
        "Specific_Objective_Label",
        "Policy_Objective_Label",
    ]
].rename(columns={"record_id": "source_record_id"})
benchmark_df["query_text"] = benchmark_candidates.apply(
    lambda row: strip_label_clues(row["search_text"], row), axis=1
)

# rearrange columns to have source_record_id and query_text first
benchmark_df = benchmark_df[
    ["source_record_id", "query_text"]
    + [
        column
        for column in benchmark_df.columns
        if column not in ["source_record_id", "query_text"]
    ]
]
benchmark_ids = set(benchmark_df["source_record_id"])

# a boolean mask that excludes benchmark records from retrieval results during evaluation
benchmark_exclusion_mask = processed_df["record_id"].isin(benchmark_ids).to_numpy()

benchmark_path = BENCHMARK_DIR / f"benchmark_queries_{DATASET_VERSION}.csv"
benchmark_df.to_csv(benchmark_path, index=False)
benchmark_df.head()

,source_record_id,query_text,CountryCode,Programme_Name,Fund_Name,Category_Label,Specific_Objective_Label,Policy_Objective_Label
109264,https://linkedopendata.eu/entity/Q7462945,FMC: Lifelong learning and employability. The ...,PT,"Demography, Qualifications and Inclusion – PT ...",European Social Fund Plus,Support for adult education (excluding infrast...,ESO4.7,Social Europe
120610,https://linkedopendata.eu/entity/Q7419608,"Country of Vrancea, our country, your country....",RO,Technical Assistance – RO ERDF/ESF+,European Social Fund Plus,Reinforcement of the capacity of Member State ...,SPOTA,Technical Assistance
83343,https://linkedopendata.eu/entity/Q7428929,Development of export markets and internationa...,LT,EU Funds’ Investments – LT – ERDF/ESF+/CF/JTF,European Regional Development Fund,Support for innovation clusters including betw...,RSO1.3,Smarter Europe
65065,https://linkedopendata.eu/entity/Q7488118,Green Productive Investment SMEs. During the i...,GR,Competitiveness – EL – ERDF/ESF+,European Regional Development Fund,SME business development and internationalisat...,RSO1.3,Smarter Europe
55176,https://linkedopendata.eu/entity/Q7440215,Developing a Software as a service robotics so...,FI,Innovation and skills – FI – ERDF/ESF+/JTF,Just Transition Fund,SME business development and internationalisat...,JTF,Just Transition


In [24]:
# for evaluation purposes, relevance signals are defined
# based on the project metadata labels distribution
RELEVANCE_SIGNAL_FIELDS = [
    "Specific_Objective_Label",
    "Category_Label",
    "Policy_Objective_Label",
    "Fund_Name",
    "Programme_Name",
]


def label_distribution_summary(records, fields):
    rows = []
    for field in fields:
        values = records[field].fillna("").map(clean_text)
        values = values[values.str.len().gt(0)]
        counts = values.value_counts()
        top_count = int(counts.iloc[0]) if len(counts) else 0
        rows.append(
            {
                "field": field,
                "unique_values": len(counts),
                "top_value": counts.index[0] if len(counts) else "",
                "top_value_count": top_count,
                "top_value_share": top_count / len(records),
                "top_10_share": counts.head(10).sum() / len(records),
            }
        )
    return pd.DataFrame(rows)


def benchmark_match_pool_summary(records, benchmark_queries, fields):
    rows = []
    for field in fields:
        values = records[field].fillna("").map(clean_text)
        pool_sizes = []
        for _, query_row in benchmark_queries.iterrows():
            value = clean_text(query_row.get(field))
            pool_sizes.append(int((values == value).sum()) if value else 0)
        pool_sizes = pd.Series(pool_sizes)
        rows.append(
            {
                "field": field,
                "mean_pool_size": pool_sizes.mean(),
                "median_pool_size": pool_sizes.median(),
                "p75_pool_size": pool_sizes.quantile(0.75),
                "p90_pool_size": pool_sizes.quantile(0.90),
                "max_pool_size": pool_sizes.max(),
                "queries_with_over_1000_matches": int(pool_sizes.gt(1000).sum()),
                "queries_with_over_10000_matches": int(pool_sizes.gt(10000).sum()),
            }
        )
    return pd.DataFrame(rows)


def same_label(query_row, result_row, field):
    query_value = clean_text(query_row.get(field))
    return bool(query_value) and query_value == clean_text(result_row.get(field))


# assign stricter relevance grades for evaluation:
# 3 = same category and same specific objective
# 2 = same category or same specific objective
# 1 = same broad policy objective, fund or programme
# 0 = no metadata match
def relevance_grade(query_row, result_row):
    same_specific_objective = same_label(query_row, result_row, "Specific_Objective_Label")
    same_category = same_label(query_row, result_row, "Category_Label")
    same_policy_objective = same_label(query_row, result_row, "Policy_Objective_Label")
    same_fund = same_label(query_row, result_row, "Fund_Name")
    same_programme = same_label(query_row, result_row, "Programme_Name")

    if same_specific_objective and same_category:
        return 3
    if same_specific_objective or same_category:
        return 2
    if same_policy_objective or same_fund or same_programme:
        return 1
    return 0


heldout_corpus_df = processed_df[~benchmark_exclusion_mask]
relevance_label_distribution = label_distribution_summary(processed_df, RELEVANCE_SIGNAL_FIELDS)
benchmark_match_pool_distribution = benchmark_match_pool_summary(
    heldout_corpus_df, benchmark_df, RELEVANCE_SIGNAL_FIELDS
)

display(relevance_label_distribution)
display(benchmark_match_pool_distribution)

,field,unique_values,top_value,top_value_count,top_value_share,top_10_share
0,Specific_Objective_Label,65,RSO1.3,23456,0.187666,0.714517
1,Category_Label,619,SME business development and internationalisat...,16914,0.135325,0.447243
2,Policy_Objective_Label,10,Social Europe,56326,0.450651,0.998800
3,Fund_Name,7,European Regional Development Fund,62581,0.500696,1.000000
4,Programme_Name,173,Competitiveness – EL – ERDF/ESF+,12902,0.103226,0.494535


,field,mean_pool_size,median_pool_size,p75_pool_size,p90_pool_size,max_pool_size,queries_with_over_1000_matches,queries_with_over_10000_matches
0,Specific_Objective_Label,9396.820,6607.5,12605.00,23415.0,23415,188,58
1,Category_Label,4550.715,2331.0,5088.00,16883.0,16883,146,31
2,Policy_Objective_Label,39403.795,39602.0,56240.00,56240.0,56240,200,176
3,Fund_Name,53129.580,57431.0,62481.00,62481.0,62481,198,182
4,Programme_Name,5347.695,3725.0,8498.25,12876.0,12876,151,50


In [25]:
# convert relevance grades 0-3 into gains 0, 1, 3, 7 for NDCG scoring
def ndcg_at_k(grades, k):
    gains = np.asarray([2**grade - 1 for grade in grades[:k]], dtype=float)
    if gains.size == 0 or np.all(gains == 0):
        return 0.0
    if gains.size == 1:
        return 1.0
    ranking_scores = np.arange(len(gains), 0, -1)
    return float(ndcg_score(np.asarray([gains]), np.asarray([ranking_scores]), k=k))


def confidence_weights_label(weights):
    return (
        f"semantic={weights['semantic']:.2f}, "
        f"keyword={weights['keyword']:.2f}, "
        f"geographic={weights['geographic']:.2f}"
    )


# evaluates each benchmark query separately so weak cases can be inspected directly
def evaluate_retriever_diagnostics(
    benchmark_queries,
    top_k=TOP_K,
    dense_candidate=DEFAULT_DENSE_CANDIDATE,
    confidence_weights=None,
    exclusion_mask=None,
):
    weights = confidence_weights or DEFAULT_CONFIDENCE_WEIGHTS
    diagnostic_rows = []

    for _, query_row in benchmark_queries.iterrows():
        results = retrieve_similar_projects(
            query_row["query_text"],
            location_text=query_row.get("CountryCode", ""),
            top_k=top_k,
            dense_candidate=dense_candidate,
            confidence_weights=weights,
            exclusion_mask=exclusion_mask,
        )
        grades = [relevance_grade(query_row, result_row) for _, result_row in results.iterrows()]
        relevant = [grade >= 2 for grade in grades]
        first_relevant_rank = next(
            (index + 1 for index, value in enumerate(relevant) if value), None
        )
        diagnostic_rows.append(
            {
                "candidate": dense_candidate,
                "confidence_weights": weights,
                "confidence_weights_label": confidence_weights_label(weights),
                "source_record_id": query_row["source_record_id"],
                "query_country": query_row.get("CountryCode", ""),
                "query_fund": query_row.get("Fund_Name", ""),
                "query_policy_objective": query_row.get("Policy_Objective_Label", ""),
                "query_specific_objective": query_row.get("Specific_Objective_Label", ""),
                "precision_at_k": sum(relevant) / top_k,
                "reciprocal_rank": 1 / first_relevant_rank if first_relevant_rank else 0.0,
                "first_relevant_rank": first_relevant_rank,
                "ndcg_at_k": ndcg_at_k(grades, top_k),
                "relevant_count": sum(relevant),
                "top_result_ids": results["Operation_Unique_Identifier"].tolist(),
                "top_result_grades": grades,
                "top_result_confidences": results["confidence"].round(6).tolist(),
                "top_k": top_k,
            }
        )

    return pd.DataFrame(diagnostic_rows)


def summarize_evaluation_diagnostics(evaluation_diagnostics):
    summary = (
        evaluation_diagnostics.groupby(["candidate", "confidence_weights_label"], as_index=False)
        .agg(
            confidence_weights=("confidence_weights", "first"),
            precision_at_k=("precision_at_k", "mean"),
            mrr=("reciprocal_rank", "mean"),
            ndcg_at_k=("ndcg_at_k", "mean"),
            query_count=("source_record_id", "count"),
            top_k=("top_k", "first"),
        )
        .sort_values("ndcg_at_k", ascending=False, ignore_index=True)
    )
    return summary[
        [
            "candidate",
            "confidence_weights",
            "confidence_weights_label",
            "precision_at_k",
            "mrr",
            "ndcg_at_k",
            "query_count",
            "top_k",
        ]
    ]


def aggregate_evaluation_by_candidate(evaluation_summary):
    aggregates = (
        evaluation_summary.groupby("candidate", as_index=False)
        .agg(
            precision_at_k_mean=("precision_at_k", "mean"),
            precision_at_k_std=("precision_at_k", "std"),
            precision_at_k_max=("precision_at_k", "max"),
            mrr_mean=("mrr", "mean"),
            mrr_std=("mrr", "std"),
            mrr_max=("mrr", "max"),
            ndcg_at_k_mean=("ndcg_at_k", "mean"),
            ndcg_at_k_std=("ndcg_at_k", "std"),
            ndcg_at_k_max=("ndcg_at_k", "max"),
            weight_count=("confidence_weights_label", "nunique"),
        )
        .sort_values("ndcg_at_k_max", ascending=False, ignore_index=True)
    )
    best_weights = (
        evaluation_summary.sort_values("ndcg_at_k", ascending=False)
        .drop_duplicates("candidate")[["candidate", "confidence_weights_label"]]
        .rename(columns={"confidence_weights_label": "best_confidence_weights_label"})
    )
    return aggregates.merge(best_weights, on="candidate", how="left")


def aggregate_evaluation_by_weight(evaluation_summary):
    return (
        evaluation_summary.groupby("confidence_weights_label", as_index=False)
        .agg(
            confidence_weights=("confidence_weights", "first"),
            precision_at_k_mean=("precision_at_k", "mean"),
            precision_at_k_std=("precision_at_k", "std"),
            precision_at_k_max=("precision_at_k", "max"),
            mrr_mean=("mrr", "mean"),
            mrr_std=("mrr", "std"),
            mrr_max=("mrr", "max"),
            ndcg_at_k_mean=("ndcg_at_k", "mean"),
            ndcg_at_k_std=("ndcg_at_k", "std"),
            ndcg_at_k_max=("ndcg_at_k", "max"),
            candidate_count=("candidate", "nunique"),
        )
        .sort_values("ndcg_at_k_mean", ascending=False, ignore_index=True)
    )


# evaluates the retriever on the benchmark queries and returns average Precision@K, MRR and NDCG@K
def evaluate_retriever(
    benchmark_queries,
    top_k=TOP_K,
    dense_candidate=DEFAULT_DENSE_CANDIDATE,
    confidence_weights=None,
    exclusion_mask=None,
):
    diagnostics = evaluate_retriever_diagnostics(
        benchmark_queries,
        top_k=top_k,
        dense_candidate=dense_candidate,
        confidence_weights=confidence_weights,
        exclusion_mask=exclusion_mask,
    )
    return summarize_evaluation_diagnostics(diagnostics).iloc[0].to_dict()


def evaluate_weight_grid_diagnostics(
    benchmark_queries,
    top_k=TOP_K,
    dense_candidates=None,
    exclusion_mask=None,
):
    dense_candidates = list(DENSE_CANDIDATES) if dense_candidates is None else dense_candidates
    return pd.concat(
        [
            evaluate_retriever_diagnostics(
                benchmark_queries,
                top_k=top_k,
                dense_candidate=dense_candidate,
                confidence_weights=weights,
                exclusion_mask=exclusion_mask,
            )
            for dense_candidate in dense_candidates
            for weights in CONFIDENCE_WEIGHT_CANDIDATES
        ],
        ignore_index=True,
    )


def evaluate_weight_grid(
    benchmark_queries,
    top_k=TOP_K,
    dense_candidates=None,
    exclusion_mask=None,
):
    diagnostics = evaluate_weight_grid_diagnostics(
        benchmark_queries,
        top_k=top_k,
        dense_candidates=dense_candidates,
        exclusion_mask=exclusion_mask,
    )
    return summarize_evaluation_diagnostics(diagnostics)


benchmark_evaluation = (
    BENCHMARK_DIR / f"benchmark_queries_evaluation_{DATASET_VERSION}_top{TOP_K}.csv"
)

if benchmark_evaluation.exists():
    evaluation_diagnostics = pd.read_csv(benchmark_evaluation)
    print(f"Loaded existing evaluation from {benchmark_evaluation}.")
else:
    evaluation_diagnostics = evaluate_weight_grid_diagnostics(
        benchmark_df,
        top_k=TOP_K,
        exclusion_mask=benchmark_exclusion_mask,
    )
    evaluation_diagnostics.to_csv(benchmark_evaluation, index=False)

evaluation_summary = summarize_evaluation_diagnostics(evaluation_diagnostics)
candidate_aggregates = aggregate_evaluation_by_candidate(evaluation_summary)
weight_aggregates = aggregate_evaluation_by_weight(evaluation_summary)

display(evaluation_summary)
display(candidate_aggregates)
display(weight_aggregates)

Loaded existing evaluation from /Users/riccardo/Projects/luiss-xai/data/benchmarks/benchmark_queries_evaluation_v1_top10.csv.


,candidate,confidence_weights,confidence_weights_label,precision_at_k,mrr,ndcg_at_k,query_count,top_k
0,multilingual_e5,"{'geographic': 0.15, 'keyword': 0.35, 'semanti...","semantic=0.50, keyword=0.35, geographic=0.15",0.8225,0.918472,0.940790,200,10
1,multilingual_e5,"{'geographic': 0.1, 'keyword': 0.3, 'semantic'...","semantic=0.60, keyword=0.30, geographic=0.10",0.8185,0.916556,0.940146,200,10
2,multilingual_e5,"{'geographic': 0.2, 'keyword': 0.3, 'semantic'...","semantic=0.50, keyword=0.30, geographic=0.20",0.8220,0.921548,0.939532,200,10
3,multilingual_e5,"{'geographic': 0.15, 'keyword': 0.25, 'semanti...","semantic=0.60, keyword=0.25, geographic=0.15",0.8195,0.918881,0.938811,200,10
4,sbert_mpnet,"{'geographic': 0.15, 'keyword': 0.35, 'semanti...","semantic=0.50, keyword=0.35, geographic=0.15",0.8210,0.929673,0.937318,200,10
5,sbert_mpnet,"{'geographic': 0.2, 'keyword': 0.3, 'semantic'...","semantic=0.50, keyword=0.30, geographic=0.20",0.8185,0.925556,0.935174,200,10
6,multilingual_e5,"{'geographic': 0.1, 'keyword': 0.2, 'semantic'...","semantic=0.70, keyword=0.20, geographic=0.10",0.8170,0.919067,0.933576,200,10
7,sbert_mpnet,"{'geographic': 0.15, 'keyword': 0.25, 'semanti...","semantic=0.60, keyword=0.25, geographic=0.15",0.8240,0.918798,0.933141,200,10
8,sbert_mpnet,"{'geographic': 0.1, 'keyword': 0.2, 'semantic'...","semantic=0.70, keyword=0.20, geographic=0.10",0.8275,0.923853,0.932370,200,10
9,sbert_mpnet,"{'geographic': 0.1, 'keyword': 0.3, 'semantic'...","semantic=0.60, keyword=0.30, geographic=0.10",0.8250,0.912056,0.929797,200,10


,candidate,precision_at_k_mean,precision_at_k_std,precision_at_k_max,mrr_mean,mrr_std,mrr_max,ndcg_at_k_mean,ndcg_at_k_std,ndcg_at_k_max,weight_count,best_confidence_weights_label
0,multilingual_e5,0.8199,0.002329,0.8225,0.918905,0.001783,0.921548,0.938571,0.002887,0.940790,5,"semantic=0.50, keyword=0.35, geographic=0.15"
1,sbert_mpnet,0.8232,0.003511,0.8275,0.921987,0.006784,0.929673,0.933560,0.002849,0.937318,5,"semantic=0.50, keyword=0.35, geographic=0.15"


,confidence_weights_label,confidence_weights,precision_at_k_mean,precision_at_k_std,precision_at_k_max,mrr_mean,mrr_std,mrr_max,ndcg_at_k_mean,ndcg_at_k_std,ndcg_at_k_max,candidate_count
0,"semantic=0.50, keyword=0.35, geographic=0.15","{'geographic': 0.15, 'keyword': 0.35, 'semanti...",0.82175,0.001061,0.8225,0.924072,0.007920,0.929673,0.939054,0.002455,0.940790,2
1,"semantic=0.50, keyword=0.30, geographic=0.20","{'geographic': 0.2, 'keyword': 0.3, 'semantic'...",0.82025,0.002475,0.8220,0.923552,0.002834,0.925556,0.937353,0.003082,0.939532,2
2,"semantic=0.60, keyword=0.25, geographic=0.15","{'geographic': 0.15, 'keyword': 0.25, 'semanti...",0.82175,0.003182,0.8240,0.918839,0.000059,0.918881,0.935976,0.004009,0.938811,2
3,"semantic=0.60, keyword=0.30, geographic=0.10","{'geographic': 0.1, 'keyword': 0.3, 'semantic'...",0.82175,0.004596,0.8250,0.914306,0.003182,0.916556,0.934972,0.007318,0.940146,2
4,"semantic=0.70, keyword=0.20, geographic=0.10","{'geographic': 0.1, 'keyword': 0.2, 'semantic'...",0.82225,0.007425,0.8275,0.921460,0.003384,0.923853,0.932973,0.000853,0.933576,2


In [ ]:
# https://ollama.com/library/gemma4:e4b
LOCAL_LLM_MODEL_VERSION = "gemma4:e4b"
OLLAMA_GENERATE_URL = "http://localhost:11434/api/generate"
LOCAL_LLM_TIMEOUT_SECONDS = 120
PROMPT_TEMPLATE_VERSION = "xai_v1"

POSITIONING_SUGGESTION_FIELDS = {
    "Programme_Name": "programme_name",
    "Fund_Name": "fund_name",
    "Category_Label": "category_label",
    "Specific_Objective_Label": "specific_objective_label",
    "Policy_Objective_Label": "policy_objective_label",
}
BUDGET_SUGGESTION_FIELDS = {
    "Total_Eligible_Expenditure_amount": "total_eligible_expenditure_amount",
    "Project_EU_Budget": "project_eu_budget",
}

# merge the fields into a single dictionary
MATCH_METADATA_FIELDS = POSITIONING_SUGGESTION_FIELDS | BUDGET_SUGGESTION_FIELDS

EXPLANATION_PROMPT_TEMPLATE = """
You are an explanation assistant for an EU funding retrieval system.

The ranking model has already selected this historical project. Only explain why it was matched.

Rules:
- Use only the provided JSON input.
- Do not change the project rank or confidence score.
- Do not invent programme, fund, category, objective, budget, or location information.
- Do not claim that the user project is eligible for funding.
- Explain the match in plain English for a non-technical user.
- Do not include internal reasoning steps or notes.
- If geography did not contribute to the score, say that geography was not used.
- Return valid JSON only, with no markdown.

Input JSON:
{{llm_input_json}}

Return this JSON structure:
{
  "project_id": "...",
  "rank": 1,
  "confidence_score": 0.0,
  "explanation": "...",
  "geography_note": "..."
}
"""


# this function converts values to be JSON serializable
def json_ready_value(value):
    if value is None:
        return None
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return None if np.isnan(value) else float(value)
    if isinstance(value, float):
        return None if np.isnan(value) else value
    if pd.isna(value):
        return None
    if isinstance(value, str):
        text = clean_text(value)
        return text if text else None
    return value


def build_llm_input(input_text, location_text, result_row):
    search_text = clean_text(result_row.get("search_text", ""))
    title_summary_parts = search_text.split(". ", 1)
    title = title_summary_parts[0]
    summary = title_summary_parts[1] if len(title_summary_parts) > 1 else search_text

    return {
        "user_project": {
            "text": clean_text(input_text),
            "location": json_ready_value(location_text),
        },
        "matched_project": {
            "project_id": json_ready_value(result_row.get("Operation_Unique_Identifier")),
            "rank": int(result_row["rank"]),
            "title": title,
            "summary": summary,
            "country": json_ready_value(result_row.get("Country")),
            "nuts3_label": json_ready_value(result_row.get("NUTS3_Label")),
            "lau_labels": json_ready_value(result_row.get("LAU_Labels")),
        },
        "matched_project_metadata": {
            output_name: json_ready_value(result_row.get(source_name))
            for source_name, output_name in MATCH_METADATA_FIELDS.items()
        },
        "scores": {
            "semantic_score": json_ready_value(result_row.get("semantic_score")),
            "keyword_score": json_ready_value(result_row.get("keyword_score")),
            "geographic_score": json_ready_value(result_row.get("geographic_score")),
            "confidence_score": json_ready_value(result_row.get("confidence")),
        },
    }


def build_explanation_prompt(llm_input):
    llm_input_json = json.dumps(llm_input, indent=2, ensure_ascii=False)
    return EXPLANATION_PROMPT_TEMPLATE.replace("{{llm_input_json}}", llm_input_json)


# https://docs.ollama.com/api/generate
def request_local_llm(prompt, model_name=LOCAL_LLM_MODEL_VERSION):
    payload = {
        "model": model_name,
        "prompt": prompt,
        "stream": False,
        "format": "json",
        "options": {"temperature": 0},
    }
    request = Request(
        OLLAMA_GENERATE_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urlopen(request, timeout=LOCAL_LLM_TIMEOUT_SECONDS) as response:
            return json.load(response)
    except (HTTPError, URLError, TimeoutError, json.JSONDecodeError) as error:
        raise RuntimeError("Ollama is not working :(") from error


def validate_llm_output(llm_output, llm_input):
    required_fields = [
        "project_id",
        "rank",
        "confidence_score",
        "explanation",
        "geography_note",
    ]
    missing_fields = [field for field in required_fields if field not in llm_output]
    if missing_fields:
        raise ValueError(f"Local LLM output is missing fields: {', '.join(missing_fields)}")

    expected_project_id = llm_input["matched_project"]["project_id"]
    expected_rank = llm_input["matched_project"]["rank"]
    expected_confidence = llm_input["scores"]["confidence_score"]
    if (
        llm_output["project_id"] != expected_project_id
        or int(llm_output["rank"]) != expected_rank
        or not np.isclose(float(llm_output["confidence_score"]), float(expected_confidence))
    ):
        raise ValueError("Local LLM output has different project_id, rank or confidence_score")

    explanation = clean_text(llm_output["explanation"])
    geography_note = clean_text(llm_output["geography_note"])
    if not explanation or not geography_note:
        raise ValueError("Local LLM returned an empty explanation or geography_note")

    return {
        "project_id": expected_project_id,
        "rank": expected_rank,
        "confidence_score": expected_confidence,
        "explanation": explanation,
        "geography_note": geography_note,
    }


def generate_match_explanation(llm_input, model_name=LOCAL_LLM_MODEL_VERSION):
    prompt = build_explanation_prompt(llm_input)
    response = request_local_llm(prompt, model_name=model_name)
    try:
        llm_output = json.loads(response["response"])
    except (KeyError, TypeError, json.JSONDecodeError) as error:
        raise ValueError("Local LLM response does not match the expected JSON") from error
    return validate_llm_output(llm_output, llm_input)


def summarize_suggestions(input_text, location_text, results):
    positioning_suggestions = {}
    for source_name, output_name in POSITIONING_SUGGESTION_FIELDS.items():
        values = results[source_name].dropna().map(clean_text)
        values = values[values.str.len() > 0]
        positioning_suggestions[output_name] = [
            {"value": value, "match_count": int(count)}
            for value, count in values.value_counts().head(3).items()
        ]

    budget_benchmarks = {}
    for source_name, output_name in BUDGET_SUGGESTION_FIELDS.items():
        numeric_values = pd.to_numeric(results[source_name], errors="coerce").dropna()
        budget_benchmarks[output_name] = {
            "median": float(numeric_values.median()) if len(numeric_values) else None,
            "p25": float(numeric_values.quantile(0.25)) if len(numeric_values) else None,
            "p75": float(numeric_values.quantile(0.75)) if len(numeric_values) else None,
        }
    positioning_suggestions["budget_benchmarks"] = budget_benchmarks

    llm_inputs = [
        build_llm_input(input_text, location_text, result_row)
        for _, result_row in results.iterrows()
    ]
    llm_outputs = [generate_match_explanation(llm_input) for llm_input in llm_inputs]

    return {
        "positioning_suggestions": positioning_suggestions,
        "llm_inputs": llm_inputs,
        "llm_outputs": llm_outputs,
    }


def suggestion_summary_table(suggestion):
    top_llm_input = max(
        suggestion["llm_inputs"],
        key=lambda llm_input: llm_input["scores"]["confidence_score"] or 0,
    )
    matched_project = top_llm_input["matched_project"]
    metadata = top_llm_input["matched_project_metadata"]
    scores = top_llm_input["scores"]
    explanation = next(
        (
            output
            for output in suggestion["llm_outputs"]
            if output["project_id"] == matched_project["project_id"]
        ),
        {},
    )
    budget_type = "total_eligible_expenditure_amount"
    budget_benchmark = suggestion["positioning_suggestions"]["budget_benchmarks"][budget_type]

    return pd.DataFrame(
        [
            {
                "title": matched_project["title"],
                "country": matched_project["country"],
                "programme": metadata["programme_name"],
                "fund": metadata["fund_name"],
                "confidence": scores["confidence_score"],
                "explanation": explanation.get("explanation"),
                "geography_note": explanation.get("geography_note"),
                "budget_p25": budget_benchmark["p25"],
                "budget_median": budget_benchmark["median"],
                "budget_p75": budget_benchmark["p75"],
            }
        ]
    )


suggestions = summarize_suggestions(
    example_query,
    "Rovereto, Italy",
    example_results,
)

suggestion_table = suggestion_summary_table(suggestions)
suggestion_table

,title,country,programme,fund,confidence,explanation,geography_note,p25,median,p75
0,S4T,Italy,European Urban Initiative 2021-2027,European Regional Development Fund,0.999858,The project was matched because both the user ...,"The match was supported by the location, as bo...",296772.96,1460220.63,2465693.25


In [27]:
def stable_hash(value):
    payload = json.dumps(value, sort_keys=True, default=str, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def latest_block_hash():
    blocks = sorted(AUDIT_DIR.glob("*.json"))
    if not blocks:
        return "0" * 64
    with open(blocks[-1], encoding="utf-8") as file:
        return json.load(file)["current_block_hash"]


def write_audit_block(input_text, location_metadata, results, suggestions):
    llm_inputs = suggestions.get("llm_inputs", [])
    llm_outputs = suggestions.get("llm_outputs", [])
    positioning_suggestions = suggestions.get("positioning_suggestions", suggestions)

    block_without_hash = {
        "run_id": str(uuid.uuid4()),
        "timestamp": datetime.now(UTC).isoformat(),
        "input_text_hash": stable_hash(clean_text(input_text)),
        "uploaded_file_hash": None,
        "location_metadata_hash": (stable_hash(location_metadata) if location_metadata else None),
        "model_version": MODEL_VERSION,
        "dataset_version": DATASET_VERSION,
        "top_k_result_ids": results["Operation_Unique_Identifier"].tolist(),
        "llm_input_hash": stable_hash(llm_inputs) if llm_inputs else None,
        "prompt_template_version": PROMPT_TEMPLATE_VERSION if llm_inputs else None,
        "llm_model_version": LOCAL_LLM_MODEL_VERSION if llm_outputs else None,
        "explanation_hash": stable_hash(llm_outputs) if llm_outputs else None,
        "suggestions_hash": stable_hash(positioning_suggestions),
        "previous_block_hash": latest_block_hash(),
    }
    block = dict(block_without_hash)
    block["current_block_hash"] = stable_hash(block_without_hash)
    timestamp = str(block["timestamp"]).replace(":", "-")
    block_path = AUDIT_DIR / f"{timestamp}_{block['run_id']}.json"
    with open(block_path, "w", encoding="utf-8") as file:
        json.dump(block, file, indent=2, ensure_ascii=False)
    return block_path, block


location_metadata = geocode_location("Italy")
audit_path, audit_block = write_audit_block(
    example_query, location_metadata, example_results, suggestions
)
audit_block

{'run_id': '1f921ea9-3fb2-4b36-b78c-f9c3c3a9cd50',
 'timestamp': '2026-05-08T16:34:20.013740+00:00',
 'input_text_hash': 'bfa59f5683c9e8946b28a93da60446f45e06e36a75c7b3a95007a6aa4e0bce37',
 'uploaded_file_hash': None,
 'location_metadata_hash': 'e5eb35a7e657e340e0882183dce7502a0bd7beaadb9f44f3325d38c1019b6b1b',
 'model_version': 'hybrid_bm25_dense_geo_v1',
 'dataset_version': 'v1',
 'top_k_result_ids': ['https://linkedopendata.eu/entity/Q7420768',
  'https://linkedopendata.eu/entity/Q7414233',
  'https://linkedopendata.eu/entity/Q7411440',
  'https://linkedopendata.eu/entity/Q7411372',
  'https://linkedopendata.eu/entity/Q7411443'],
 'llm_input_hash': 'ea163e0e58117b33e8a9af655a71729cd9ba7ecc9ee6f8da4ce876e1c520cbf2',
 'prompt_template_version': 'xai_v1',
 'llm_model_version': 'gemma4:e4b',
 'explanation_hash': '5190e0bc233cffe6208a7a0fc14036729b461ce1ff1eb3ab19e932bbf8f1e1c0',
 'suggestions_hash': '68a70054542a90978b8df7013c02a5b3c19d7f29162a39d5e59646ca6e48ab75',
 'previous_block_has

In [28]:
user_project_idea = """
A municipality wants to renovate public school buildings with better insulation,
smart energy monitoring, and renewable heating systems to lower emissions and operating costs.
"""
user_location = "Italy"

user_results = retrieve_similar_projects(
    user_project_idea, location_text=user_location, top_k=TOP_K
)
user_suggestions = summarize_suggestions(
    user_project_idea,
    user_location,
    user_results,
)
user_audit_path, user_audit_block = write_audit_block(
    user_project_idea,
    geocode_location(user_location),
    user_results,
    user_suggestions,
)

user_results

,rank,confidence,semantic_score,keyword_score,geographic_score,Operation_Unique_Identifier,Programme_Name,Fund_Name,Category_Label,Specific_Objective_Label,Policy_Objective_Label,Total_Eligible_Expenditure_amount,Project_EU_Budget,Country,NUTS3_Label,LAU_Labels,InfoRegio_URL,search_text
58808,1,0.832527,1.000000,0.662636,0.0,https://linkedopendata.eu/entity/Q7356190,Auvergne-Rhône-Alpes - ERDF/ESF+/JTF,European Regional Development Fund,Energy efficiency renovation or energy efficie...,RSO2.1,"Greener, carbon-free Europe",211519.00,51000.00,France,Haute-Loire,Chaspinhac,NaN,Energy renovation of local public buildings. T...
120368,2,0.766188,0.808840,1.000000,0.0,https://linkedopendata.eu/entity/Q7417564,South-West Oltenia – RO – ERDF,European Regional Development Fund,Energy efficiency renovation or energy efficie...,RSO2.1,"Greener, carbon-free Europe",661421.18,648192.75,Romania,Mehedinţi,NaN,NaN,Increasing energy efficiency at Jidostița Seco...
20265,3,0.754234,0.854789,0.779410,0.0,https://linkedopendata.eu/entity/Q7408118,Environment - CZ - ERDF/CF,European Regional Development Fund,Energy efficiency renovation or energy efficie...,RSO2.1,"Greener, carbon-free Europe",133282.10,71265.94,Czechia,Hradec Králové Region,Rychnovek,NaN,Building modifications of the elementary schoo...
34336,4,0.746119,0.864622,0.704418,0.0,https://linkedopendata.eu/entity/Q7408498,Environment - CZ - ERDF/CF,European Regional Development Fund,Energy efficiency renovation or energy efficie...,RSO2.1,"Greener, carbon-free Europe",1413238.64,909984.36,Czechia,Zlín Region,Valašské Meziříčí,NaN,Improvement of the thermal and technical prope...
96465,5,0.741109,0.773401,0.998639,0.0,https://linkedopendata.eu/entity/Q7375718,Podkarpacie - PL - ERDF/ESF+,European Regional Development Fund,Energy efficiency renovation or energy efficie...,RSO2.1,"Greener, carbon-free Europe",991418.95,384000.00,Poland,Krośnieński,Zagórz,NaN,Improving the energy efficiency of public buil...
22373,6,0.737524,0.836455,0.760028,0.0,https://linkedopendata.eu/entity/Q7408165,Environment - CZ - ERDF/CF,European Regional Development Fund,Energy efficiency renovation or energy efficie...,RSO2.1,"Greener, carbon-free Europe",4363376.17,2316952.75,Czechia,Plzeň Region,Plzeň,NaN,Energy solution of VOŠ and SPŠE buildings. The...
58015,7,0.735695,0.871118,0.629558,0.0,https://linkedopendata.eu/entity/Q7356506,Auvergne-Rhône-Alpes - ERDF/ESF+/JTF,European Regional Development Fund,Energy efficiency renovation or energy efficie...,RSO2.1,"Greener, carbon-free Europe",401839.20,102872.67,France,Loire,Chenereilles,NaN,Energy renovation of school and administrative...
26281,8,0.735207,0.831475,0.765874,0.0,https://linkedopendata.eu/entity/Q7408658,Environment - CZ - ERDF/CF,European Regional Development Fund,Energy efficiency renovation or energy efficie...,RSO2.1,"Greener, carbon-free Europe",1156714.16,654815.89,Czechia,South Bohemian Region,Soběslav,NaN,Reduction of energy performance of public buil...
33300,9,0.731681,0.794746,0.876794,0.0,https://linkedopendata.eu/entity/Q7408097,Environment - CZ - ERDF/CF,European Regional Development Fund,Energy efficiency renovation or energy efficie...,RSO2.1,"Greener, carbon-free Europe",73644.88,49474.63,Czechia,Hradec Králové Region,Rychnovek,NaN,Complex austerity measures - OÚ Rychnovek. The...
59127,10,0.730735,0.880008,0.573645,0.0,https://linkedopendata.eu/entity/Q7356344,Auvergne-Rhône-Alpes - ERDF/ESF+/JTF,European Regional Development Fund,Energy efficiency renovation or energy efficie...,RSO2.1,"Greener, carbon-free Europe",950583.27,693926.00,France,Haute-Savoie,Neydens,NaN,Renovation of Puy-Saint-Martin Elementary Scho...
